In [ ]:
import json
from collections import Counter
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

DRIVE = Path('/content/drive/MyDrive')

def load_idx(name):
    """Load prediction file and key by idx."""
    recs = json.loads((DRIVE / name).read_text())
    return {r['idx']: r for r in recs}

def partition(mid_files, frontier_file, label):
    mids = [load_idx(f) for f in mid_files]
    frontier = load_idx(frontier_file)
    common = set(mids[0].keys())
    for m in mids[1:]:
        common &= set(m.keys())
    common &= set(frontier.keys())

    high_risk = {'n': 0, 'wrong': 0}
    confident = {'n': 0, 'wrong': 0}
    three_of_four = {'n': 0, 'wrong': 0}
    other_disagree = {'n': 0, 'wrong': 0}

    for i in common:
        preds = [m[i]['pred'] for m in mids]
        gold = mids[0][i]['gold']
        front = frontier[i]['pred']
        counts = Counter(preds)
        top, top_n = counts.most_common(1)[0]

        if top_n == 4:
            if front != top:
                high_risk['n'] += 1
                if top != gold: high_risk['wrong'] += 1
            else:
                confident['n'] += 1
                if top != gold: confident['wrong'] += 1
        elif top_n == 3:
            three_of_four['n'] += 1
            if top != gold: three_of_four['wrong'] += 1
        else:
            other_disagree['n'] += 1
            if top != gold: other_disagree['wrong'] += 1

    def rate(d): return d['wrong'] / d['n'] if d['n'] else 0.0

    print(f"\n=== {label} (n_common={len(common)}) ===")
    print(f"  HIGH_RISK     (4-of-4 mid agree, frontier dissents): n={high_risk['n']:4d}, wrong rate = {rate(high_risk)*100:5.1f}%")
    print(f"  CONFIDENT     (4-of-4 mid agree, frontier agrees):   n={confident['n']:4d}, wrong rate = {rate(confident)*100:5.1f}%")
    print(f"  3-of-4        (vote-share = 0.75):                   n={three_of_four['n']:4d}, wrong rate = {rate(three_of_four)*100:5.1f}%")
    print(f"  other_disagree(2-2 or 2-1-1 splits):                 n={other_disagree['n']:4d}, wrong rate = {rate(other_disagree)*100:5.1f}%")
    print(f"  Total: {high_risk['n']+confident['n']+three_of_four['n']+other_disagree['n']}")
    print(f"  Vote-share = 0.75 gap: HIGH_RISK {rate(high_risk)*100:.1f}% vs 3-of-4 {rate(three_of_four)*100:.1f}% = {(rate(high_risk)-rate(three_of_four))*100:.1f} pp")

    return {
        'high_risk': {**high_risk, 'wrong_rate': rate(high_risk)},
        'confident': {**confident, 'wrong_rate': rate(confident)},
        'three_of_four': {**three_of_four, 'wrong_rate': rate(three_of_four)},
        'other_disagree': {**other_disagree, 'wrong_rate': rate(other_disagree)},
        'vote_share_gap_pp': (rate(high_risk) - rate(three_of_four)) * 100,
    }

print("Paper claims:")
print("  MedQA   HIGH_RISK: n=102, wrong 96.1% | 3-of-4: n=325, wrong 12.9% | gap 83.2 pp")
print("  MedMCQA HIGH_RISK: n=134, wrong 82.8% | 3-of-4: n=502, wrong 24.3% | gap 58.5 pp")

medqa = partition(
    mid_files=['flan_t5_results.json', 'llama_results.json',
               'qwen_results.json', 'gemma3n_results.json'],
    frontier_file='medqa_gpt4o_results.json',
    label='MedQA',
)

medmcqa = partition(
    mid_files=['medmcqa_flan_t5_results.json', 'medmcqa_llama_results.json',
               'medmcqa_qwen_results.json', 'medmcqa_gemma3n_results.json'],
    frontier_file='medmcqa_gpt4o_results.json',
    label='MedMCQA',
)

output = {
    'experiment': 'confidence_calibration_proxy',
    'note': 'Vote-share used as confidence proxy (logprobs not cached). 4 mid-tier models (Flan, Llama, Qwen, Gemma) + GPT-4o frontier.',
    'finding': 'A vote-share >= 0.75 confidence threshold lumps together HIGH_RISK (4-of-4 mid agree, frontier dissents) and 3-of-4 (one mid-tier dissents), but these cohorts have very different wrong rates. Vote-share alone is not a useful confidence proxy.',
    'rules': {
        'high_risk':      'All 4 mid-tier models agree on X AND frontier picks Y != X',
        'confident':      'All 4 mid-tier models agree AND frontier agrees',
        'three_of_four':  'Exactly 3 of 4 mid-tier models agree (vote-share = 0.75)',
        'other_disagree': '2-2 or 2-1-1 split among mid-tier',
    },
    'medqa': medqa,
    'medmcqa': medmcqa,
}

out = DRIVE / 'exp9_confidence_proxy.json'
out.write_text(json.dumps(output, indent=2))
print(f"\nSaved {out} ({out.stat().st_size} bytes)")

In [ ]:
import json
from collections import Counter
from pathlib import Path

DRIVE = Path('/content/drive/MyDrive')

files = ['flan_t5_results.json', 'llama_results.json', 'qwen_results.json',
         'gemma3n_results.json', 'medqa_gpt4o_results.json', 'deepseek_results.json']

for name in files:
    recs = json.loads((DRIVE / name).read_text())
    print(f"\n=== {name} (n={len(recs)}) ===")
    print(f"  Keys in record: {list(recs[0].keys())}")
    print(f"  First 2 records: {recs[:2]}")
    preds = Counter(r.get('pred') for r in recs)
    print(f"  Pred distribution: {dict(preds)}")
    print(f"  Null preds: {sum(1 for r in recs if r.get('pred') is None)}")

In [ ]:
import json
from collections import Counter
from pathlib import Path

DRIVE = Path('/content/drive/MyDrive')

def load_idx(name):
    return {r['idx']: r for r in json.loads((DRIVE / name).read_text())}

def partition(mid_files, frontier_file, label):
    mids = [load_idx(f) for f in mid_files]
    frontier = load_idx(frontier_file)
    common = set(mids[0].keys())
    for m in mids[1:]:
        common &= set(m.keys())
    common &= set(frontier.keys())

    common = {i for i in common
              if all(m[i]['pred'] is not None for m in mids)
              and frontier[i]['pred'] is not None}

    confident      = {'n': 0, 'wrong': 0}
    high_risk      = {'n': 0, 'wrong': 0}
    other_3of4     = {'n': 0, 'wrong': 0}
    disagreement   = {'n': 0, 'wrong': 0}

    for i in common:
        mid_preds = [m[i]['pred'] for m in mids]
        gold = mids[0][i]['gold']
        front = frontier[i]['pred']
        all4 = mid_preds + [front]
        counts = Counter(all4)
        top, top_n = counts.most_common(1)[0]

        if top_n == 4:
            bucket = confident
        elif top_n == 3:
            bucket = other_3of4 if front == top else high_risk
        else:
            bucket = disagreement

        bucket['n'] += 1
        if top != gold:
            bucket['wrong'] += 1

    def rate(d): return d['wrong'] / d['n'] if d['n'] else 0.0

    print(f"\n=== {label} (n_common={len(common)}) ===")
    print(f"  CONFIDENT    (vote-share 1.0):                            n={confident['n']:4d}, wrong = {rate(confident)*100:5.1f}%")
    print(f"  HIGH_RISK    (vote-share 0.75, frontier dissents):        n={high_risk['n']:4d}, wrong = {rate(high_risk)*100:5.1f}%")
    print(f"  OTHER_3OF4   (vote-share 0.75, frontier in majority):     n={other_3of4['n']:4d}, wrong = {rate(other_3of4)*100:5.1f}%")
    print(f"  DISAGREEMENT (vote-share <= 0.5):                         n={disagreement['n']:4d}, wrong = {rate(disagreement)*100:5.1f}%")
    print(f"  Total: {confident['n']+high_risk['n']+other_3of4['n']+disagreement['n']}")
    print(f"  Vote-share=0.75 gap: HIGH_RISK {rate(high_risk)*100:.1f}% vs OTHER_3OF4 {rate(other_3of4)*100:.1f}% = {(rate(high_risk)-rate(other_3of4))*100:.1f} pp")

    return {
        'confident':    {**confident,    'wrong_rate': rate(confident)},
        'high_risk':    {**high_risk,    'wrong_rate': rate(high_risk)},
        'other_3of4':   {**other_3of4,   'wrong_rate': rate(other_3of4)},
        'disagreement': {**disagreement, 'wrong_rate': rate(disagreement)},
        'vote_share_gap_pp': (rate(high_risk) - rate(other_3of4)) * 100,
    }

print("Paper claims:")
print("  MedQA   HIGH_RISK: n=102, wrong 96.1% | OTHER_3OF4: n=325, wrong 12.9% | gap 83.2 pp")
print("  MedMCQA HIGH_RISK: n=134, wrong 82.8% | OTHER_3OF4: n=502, wrong 24.3% | gap 58.5 pp")

medqa = partition(
    mid_files=['llama_results.json', 'qwen_results.json', 'gemma3n_results.json'],
    frontier_file='medqa_gpt4o_results.json',
    label='MedQA',
)
medmcqa = partition(
    mid_files=['medmcqa_llama_results.json', 'medmcqa_qwen_results.json', 'medmcqa_gemma3n_results.json'],
    frontier_file='medmcqa_gpt4o_results.json',
    label='MedMCQA',
)

output = {
    'experiment': 'confidence_calibration_proxy',
    'note': 'Vote-share among 4-model ensemble (3 mid-tier + frontier) used as a confidence proxy. Mid-tier = Llama-3-8B, Qwen-2.5-7B, Gemma-3n-E4B. Frontier = GPT-4o.',
    'finding': 'A vote-share >= 0.75 threshold lumps together HIGH_RISK (3 mid agree, frontier dissents) and OTHER_3OF4 (2 mid + frontier agree, 1 mid dissents). The two cohorts have very different wrong rates, so vote-share alone is not a useful confidence proxy. The frontier-vs-consensus signal is what matters.',
    'rules': {
        'confident':    'All 4 models agree (3 mid + frontier)',
        'high_risk':    '3 mid-tier agree on X AND frontier picks Y != X',
        'other_3of4':   '2 mid-tier + frontier agree on X AND 1 mid-tier picks Y != X',
        'disagreement': 'No 3-of-4 majority among {3 mid, frontier}',
    },
    'medqa': medqa,
    'medmcqa': medmcqa,
}

out = DRIVE / 'exp9_confidence_proxy.json'
out.write_text(json.dumps(output, indent=2))
print(f"\nSaved {out} ({out.stat().st_size} bytes)")

In [ ]:
import json
from pathlib import Path
from shutil import copyfile

DRIVE = Path('/content/drive/MyDrive')

exp9 = json.loads((DRIVE / 'exp9_confidence_proxy.json').read_text())

exp1 = json.loads((DRIVE / 'exp1_holdout_cis.json').read_text())
strict_medmcqa = exp1['medmcqa_gpt4o_frontier']

exp9['note'] = (
    'Vote-share among 4-model ensemble (3 mid-tier + frontier) used as confidence proxy. '
    'Mid-tier = Llama-3-8B, Qwen-2.5-7B, Gemma-3n-E4B. Frontier = GPT-4o. '
    'MedMCQA has two reported variants: a looser rule (n=188, matches Table IV detector) '
    'and a stricter rule (n=134, used in Section V-F headline). Numbers for both are recorded; '
    'see exp1_holdout_cis.json for strict-rule source.'
)

exp9['medmcqa'] = {
    'loose_rule': {
        'description': 'Same definition as MedQA: 3 mid-tier agree + GPT-4o dissents. Matches Table IV detector validation (detector_medmcqa_validation.json: n=188, prec 83.0%).',
        **exp9['medmcqa'],
    },
    'strict_rule_v_f': {
        'description': 'Section V-F headline rule. Source: exp1_holdout_cis.json field medmcqa_gpt4o_frontier.',
        'high_risk':  {'n': strict_medmcqa['high_risk_n'],
                       'wrong': strict_medmcqa['high_risk_wrong'],
                       'wrong_rate': strict_medmcqa['high_risk_precision']},
        'confident':  {'n': strict_medmcqa['confident_n'],
                       'wrong': strict_medmcqa['confident_wrong'],
                       'wrong_rate': strict_medmcqa['confident_wrong_rate']},
        'other_3of4_paper_reported': {'n': 502, 'wrong_rate': 0.243},
        'vote_share_gap_pp_paper_reported': 58.5,
    },
}

out = DRIVE / 'exp9_confidence_proxy.json'
out.write_text(json.dumps(exp9, indent=2))
print(f"Updated {out} ({out.stat().st_size} bytes)\n")

release_target = DRIVE / 'icdm_release' / 'analysis' / 'exp9_confidence_proxy.json'
copyfile(out, release_target)
print(f"Copied to release: {release_target} ({release_target.stat().st_size} bytes)")

print("\n--- Final exp9_confidence_proxy.json contents ---")
print(json.dumps(exp9, indent=2))

In [ ]:
import json
from pathlib import Path
from scipy.stats import fisher_exact, chi2_contingency

DRIVE = Path('/content/drive/MyDrive')
data = json.loads((DRIVE / 'audit_5wrong_baseline.json').read_text())

print("Loaded audit_5wrong_baseline.json:")
print(json.dumps(data, indent=2))

def run_tests(label, payload):
    trap = payload['trap']
    all5 = payload['all_5_wrong']
    a = trap['unanim'];          b = trap['n']  - trap['unanim']
    c = all5['unanim'];          d = all5['n']  - all5['unanim']
    table = [[a, b], [c, d]]
    odds, p_fisher = fisher_exact(table, alternative='two-sided')
    chi2, p_chi2, _, _ = chi2_contingency(table)
    print(f"\n=== {label} ===")
    print(f"  Trap:       {a}/{trap['n']}  = {trap['rate']*100:.2f}%")
    print(f"  All-5-wrong: {c}/{all5['n']} = {all5['rate']*100:.2f}%")
    print(f"  Contingency table [[trap_unanim, trap_other],[all5_unanim, all5_other]] = {table}")
    print(f"  Fisher exact (two-sided): p = {p_fisher:.4f}")
    print(f"  Chi-square:               p = {p_chi2:.4f}")
    return p_fisher

print("\nPaper claims: MedQA p=0.22, MedMCQA p=0.14")
p_medqa  = run_tests('MedQA',  data['medqa'])
p_medmcqa = run_tests('MedMCQA', data['medmcqa'])

print("\n--- Verdict ---")
print(f"MedQA   recomputed p = {p_medqa:.4f}  | paper says 0.22  | match={'YES' if abs(p_medqa-0.22)<0.01 else 'NO (off by '+f'{abs(p_medqa-0.22):.3f}'+')'}")
print(f"MedMCQA recomputed p = {p_medmcqa:.4f}  | paper says 0.14  | match={'YES' if abs(p_medmcqa-0.14)<0.01 else 'NO (off by '+f'{abs(p_medmcqa-0.14):.3f}'+')'}")
if abs(p_medqa-0.14)<0.01 and abs(p_medmcqa-0.22)<0.01:
    print("\n>>> LABELS APPEAR SWAPPED. Paper should say: MedQA p=0.14, MedMCQA p=0.22 <<<")

In [ ]:
import json
from pathlib import Path
import numpy as np
from scipy.stats import ttest_ind

DRIVE = Path('/content/drive/MyDrive')
data = json.loads((DRIVE / 'programmatic_pc_scores.json').read_text())

print(f"Top-level type: {type(data).__name__}")
if isinstance(data, dict):
    print(f"Top-level keys: {list(data.keys())}")
    for k in data:
        v = data[k]
        print(f"  {k}: type={type(v).__name__}, " + (f"len={len(v)}" if hasattr(v, '__len__') else f"value={v}"))
elif isinstance(data, list):
    print(f"List length: {len(data)}")
    print(f"First record keys: {list(data[0].keys()) if data else 'empty'}")
    print(f"First 2 records:\n{json.dumps(data[:2], indent=2)}")

print("\n--- Looking for trap/non-trap labels and numeric score columns ---\n")

records = None
if isinstance(data, list) and isinstance(data[0], dict):
    records = data
elif isinstance(data, dict):
    for k, v in data.items():
        if isinstance(v, list) and v and isinstance(v[0], dict):
            print(f"Found record list under key '{k}' with {len(v)} entries; first record keys: {list(v[0].keys())}")
            if records is None:
                records = v

if records is None:
    print("Could not auto-locate per-question records. Dump first 2000 chars below:")
    print(json.dumps(data, indent=2)[:2000])
else:

    keys = list(records[0].keys())
    print(f"\nAll keys: {keys}")
    label_cols = [k for k in keys if any(tok in k.lower() for tok in ['trap', 'unanim', 'group', 'label', 'cohort'])]
    numeric_cols = [k for k in keys if isinstance(records[0].get(k), (int, float)) and not isinstance(records[0].get(k), bool)]
    print(f"Candidate label cols: {label_cols}")
    print(f"Numeric cols:        {numeric_cols}")

    print(f"\nPaper claims: t = -1.62, p = 0.107, mean diff = -0.058\n")
    for lc in label_cols:
        vals = set(r.get(lc) for r in records)
        if len(vals) != 2:
            print(f"  Skip '{lc}': not binary (values = {vals})")
            continue
        v1, v2 = sorted(vals, key=str)
        group1 = [r for r in records if r.get(lc) == v1]
        group2 = [r for r in records if r.get(lc) == v2]
        for nc in numeric_cols:
            a = np.array([r[nc] for r in group1 if r.get(nc) is not None])
            b = np.array([r[nc] for r in group2 if r.get(nc) is not None])
            if len(a) < 2 or len(b) < 2: continue
            mean_diff = a.mean() - b.mean()
            t_stat, p_val = ttest_ind(a, b, equal_var=False)
            tag = " <<< MATCHES PAPER" if abs(t_stat - (-1.62)) < 0.05 and abs(p_val - 0.107) < 0.01 else ""
            print(f"  {lc}={v1} vs {v2} | col={nc:30s} | mean_diff={mean_diff:+.4f} | t={t_stat:+.3f} | p={p_val:.4f}{tag}")

In [ ]:
import json
from pathlib import Path
import numpy as np
from scipy.stats import ttest_ind

DRIVE = Path('/content/drive/MyDrive')
records = json.loads((DRIVE / 'programmatic_pc_scores.json').read_text())

print(f"Total records: {len(records)}")
subsets = set(r['subset'] for r in records)
datasets = set(r['dataset'] for r in records)
print(f"subset values: {subsets}")
print(f"dataset values: {datasets}")

from collections import Counter
print("\nCounts per (dataset, subset):")
for k, v in sorted(Counter((r['dataset'], r['subset']) for r in records).items()):
    print(f"  {k}: {v}")

numeric_cols = ['sim_early', 'sim_middle', 'sim_late', 'pc_score']
subset_vals = sorted(subsets)

print(f"\nPaper claims: t = -1.62, p = 0.107, mean diff = -0.058")
print(f"(File audit had previously shown -0.0318 on pc_score column)\n")

def run(label, group1_filter, group2_filter, g1_name, g2_name):
    print(f"\n=== {label} ===")
    g1 = [r for r in records if group1_filter(r)]
    g2 = [r for r in records if group2_filter(r)]
    print(f"  {g1_name}: n={len(g1)}, {g2_name}: n={len(g2)}")
    for nc in numeric_cols:
        a = np.array([r[nc] for r in g1 if r.get(nc) is not None])
        b = np.array([r[nc] for r in g2 if r.get(nc) is not None])
        if len(a) < 2 or len(b) < 2:
            print(f"    {nc}: insufficient data"); continue
        diff = a.mean() - b.mean()
        t_stat, p_val = ttest_ind(a, b, equal_var=False)
        tag = ""
        if abs(t_stat + 1.62) < 0.05 and abs(p_val - 0.107) < 0.01: tag = "  <<< t & p MATCH PAPER"
        if abs(diff - (-0.058)) < 0.005: tag += "  <<< mean_diff MATCHES PAPER"
        print(f"    {nc:12s}: {g1_name}_mean={a.mean():+.4f}, {g2_name}_mean={b.mean():+.4f}, diff={diff:+.4f}, t={t_stat:+.3f}, p={p_val:.4f}{tag}")

run("ALL DATASETS: trap vs non-trap",
    lambda r: r['subset'] == 'trap',
    lambda r: r['subset'] != 'trap',
    'trap', 'non_trap')

for ds in sorted(datasets):
    run(f"{ds.upper()}: trap vs non-trap",
        lambda r, d=ds: r['subset'] == 'trap' and r['dataset'] == d,
        lambda r, d=ds: r['subset'] != 'trap' and r['dataset'] == d,
        'trap', 'non_trap')

In [ ]:
import json
from pathlib import Path
import numpy as np
from scipy.stats import ttest_ind

DRIVE = Path('/content/drive/MyDrive')
records = json.loads((DRIVE / 'programmatic_pc_scores.json').read_text())

from collections import Counter
print("Keys present per (dataset, subset):")
for key in [('medqa','trap'),('medqa','nontrap'),('medmcqa','unanimous_wrong')]:
    sample = [r for r in records if r['dataset']==key[0] and r['subset']==key[1]][:1]
    if sample:
        print(f"  {key}: {sorted(sample[0].keys())}")

numeric_cols = ['sim_early', 'sim_middle', 'sim_late', 'pc_score']

def t_test(label, g1, g2, g1_name='G1', g2_name='G2'):
    print(f"\n=== {label} ===")
    print(f"  {g1_name}: n={len(g1)}, {g2_name}: n={len(g2)}")
    for nc in numeric_cols:
        a = np.array([r[nc] for r in g1 if r.get(nc) is not None])
        b = np.array([r[nc] for r in g2 if r.get(nc) is not None])
        if len(a) < 2 or len(b) < 2:
            continue
        diff = a.mean() - b.mean()
        t_stat, p_val = ttest_ind(a, b, equal_var=False)
        tag = ""
        if abs(t_stat + 1.62) < 0.05 and abs(p_val - 0.107) < 0.01: tag += "  <<< t/p MATCH"
        if abs(diff - (-0.058)) < 0.003: tag += "  <<< mean_diff MATCHES"
        print(f"    {nc:12s} (n1={len(a)},n2={len(b)}): {g1_name}={a.mean():+.4f}, {g2_name}={b.mean():+.4f}, diff={diff:+.4f}, t={t_stat:+.3f}, p={p_val:.4f}{tag}")

print(f"\nPaper claims: t = -1.62, p = 0.107, mean diff = -0.058\n")

g1 = [r for r in records if r['subset'] in ('trap','unanimous_wrong') and r['bias_label']=='PREMATURE_CLOSURE']
g2 = [r for r in records if r['subset'] in ('trap','unanimous_wrong') and r['bias_label']!='PREMATURE_CLOSURE']
t_test("A) Pooled unanimous-wrong: PC vs non-PC", g1, g2, 'PC', 'OTHER')

g1 = [r for r in records if r['dataset']=='medqa' and r['subset']=='trap' and r['bias_label']=='PREMATURE_CLOSURE']
g2 = [r for r in records if r['dataset']=='medqa' and r['subset']=='trap' and r['bias_label']!='PREMATURE_CLOSURE']
t_test("B) MedQA trap: PC vs non-PC", g1, g2, 'PC', 'OTHER')

g1 = [r for r in records if r['dataset']=='medmcqa' and r['subset']=='unanimous_wrong' and r['bias_label']=='PREMATURE_CLOSURE']
g2 = [r for r in records if r['dataset']=='medmcqa' and r['subset']=='unanimous_wrong' and r['bias_label']!='PREMATURE_CLOSURE']
t_test("C) MedMCQA UW: PC vs non-PC", g1, g2, 'PC', 'OTHER')

g1 = [r for r in records if r['bias_label']=='PREMATURE_CLOSURE']
g2 = [r for r in records if r['bias_label']!='PREMATURE_CLOSURE']
t_test("D) ALL 526: PC vs non-PC", g1, g2, 'PC', 'OTHER')

g1 = [r for r in records if r['dataset']=='medqa' and r['subset']=='trap']
g2 = [r for r in records if r['dataset']=='medqa' and r['subset']=='nontrap']
t_test("E) MedQA: trap vs nontrap", g1, g2, 'TRAP', 'NONTRAP')

g1 = [r for r in records if r['dataset']=='medqa' and r['subset']=='trap' and r['bias_label']=='PREMATURE_CLOSURE']
g2 = [r for r in records if r['dataset']=='medqa' and r['subset']=='nontrap' and r['bias_label']=='PREMATURE_CLOSURE']
t_test("F) MedQA PC-labeled only: trap vs nontrap", g1, g2, 'PC_TRAP', 'PC_NONTRAP')

In [ ]:
import json
from pathlib import Path
import numpy as np
from scipy.stats import ttest_ind

DRIVE = Path('/content/drive/MyDrive')
records = json.loads((DRIVE / 'programmatic_pc_scores.json').read_text())

PAPER_T, PAPER_P, PAPER_DIFF = -1.62, 0.107, -0.058

def both_t(label, a, b, name):
    """Welch + Student's t."""
    if len(a) < 2 or len(b) < 2: return
    diff = a.mean() - b.mean()
    tw, pw = ttest_ind(a, b, equal_var=False)
    ts, ps = ttest_ind(a, b, equal_var=True)
    flag_w = "  <<<W matches t/p" if abs(tw-PAPER_T) < 0.03 and abs(pw-PAPER_P) < 0.01 else ""
    flag_s = "  <<<S matches t/p" if abs(ts-PAPER_T) < 0.03 and abs(ps-PAPER_P) < 0.01 else ""
    flag_d = "  <<<diff matches" if abs(diff-PAPER_DIFF) < 0.003 else ""
    print(f"  {name:50s} (n={len(a)},{len(b)}): diff={diff:+.4f}, Welch t={tw:+.3f} p={pw:.4f}, Student t={ts:+.3f} p={ps:.4f}{flag_w}{flag_s}{flag_d}")

print(f"Target: t={PAPER_T}, p={PAPER_P}, diff={PAPER_DIFF}\n")

slices = {
    'medqa trap vs medqa nontrap':
        ([r for r in records if r['dataset']=='medqa' and r['subset']=='trap'],
         [r for r in records if r['dataset']=='medqa' and r['subset']=='nontrap']),
    'medqa trap+medmcqa UW vs medqa nontrap':
        ([r for r in records if r['subset'] in ('trap','unanimous_wrong')],
         [r for r in records if r['subset']=='nontrap']),
    'all PC-labeled vs all non-PC':
        ([r for r in records if r['bias_label']=='PREMATURE_CLOSURE'],
         [r for r in records if r['bias_label']!='PREMATURE_CLOSURE']),
    'medqa PC-trap vs medqa PC-nontrap':
        ([r for r in records if r['dataset']=='medqa' and r['subset']=='trap' and r['bias_label']=='PREMATURE_CLOSURE'],
         [r for r in records if r['dataset']=='medqa' and r['subset']=='nontrap' and r['bias_label']=='PREMATURE_CLOSURE']),
    'medqa UW (trap+nontrap PC) vs anchoring':
        ([r for r in records if r['dataset']=='medqa' and r['bias_label']=='PREMATURE_CLOSURE'],
         [r for r in records if r['dataset']=='medqa' and r['bias_label']=='ANCHORING_BIAS']),
    'medmcqa PC vs medmcqa anchoring':
        ([r for r in records if r['dataset']=='medmcqa' and r['bias_label']=='PREMATURE_CLOSURE'],
         [r for r in records if r['dataset']=='medmcqa' and r['bias_label']=='ANCHORING_BIAS']),
}

print("--- pc_score across all slices (Welch + Student's) ---")
for label, (g1, g2) in slices.items():
    a = np.array([r['pc_score'] for r in g1 if r.get('pc_score') is not None])
    b = np.array([r['pc_score'] for r in g2 if r.get('pc_score') is not None])
    both_t(label, a, b, label)

print("\n--- sim_late - sim_early delta across slices (Welch + Student's) ---")
def delta(r):
    if r.get('sim_late') is None or r.get('sim_early') is None: return None
    return r['sim_late'] - r['sim_early']
for label, (g1, g2) in slices.items():
    a = np.array([delta(r) for r in g1 if delta(r) is not None])
    b = np.array([delta(r) for r in g2 if delta(r) is not None])
    both_t(label, a, b, label)

print("\n--- sim_late alone ---")
for label, (g1, g2) in slices.items():
    a = np.array([r['sim_late'] for r in g1 if r.get('sim_late') is not None])
    b = np.array([r['sim_late'] for r in g2 if r.get('sim_late') is not None])
    both_t(label, a, b, label)

In [ ]:
import json
import numpy as np
from pathlib import Path
from scipy.stats import fisher_exact, ttest_ind
from shutil import copyfile

DRIVE = Path('/content/drive/MyDrive')
RELEASE = DRIVE / 'icdm_release' / 'analysis'

audit_5w = json.loads((DRIVE / 'audit_5wrong_baseline.json').read_text())
for ds in ['medqa', 'medmcqa']:
    t = audit_5w[ds]['trap']
    a5 = audit_5w[ds]['all_5_wrong']
    _, p = fisher_exact([[t['unanim'], t['n']-t['unanim']],
                         [a5['unanim'], a5['n']-a5['unanim']]],
                         alternative='two-sided')
    audit_5w[ds]['fisher_exact_two_sided_p'] = float(p)
audit_5w['note'] = 'Fisher exact two-sided p-values added. Neither dataset reaches p<0.05, supporting the honest-null framing in Section VI-G.'
(DRIVE / 'audit_5wrong_baseline.json').write_text(json.dumps(audit_5w, indent=2))
copyfile(DRIVE / 'audit_5wrong_baseline.json', RELEASE / 'audit_5wrong_baseline.json')

records = json.loads((DRIVE / 'programmatic_pc_scores.json').read_text())
trap = np.array([r['pc_score'] for r in records if r['dataset']=='medqa' and r['subset']=='trap'])
nontrap = np.array([r['pc_score'] for r in records if r['dataset']=='medqa' and r['subset']=='nontrap'])
t_stat, p_val = ttest_ind(trap, nontrap, equal_var=False)

audit_pc = {
    "experiment": "programmatic_pc_score_trap_vs_nontrap",
    "description": "pc_score = sim_early - sim_late. Tests whether MedQA trap questions have lower pc_score (later convergence to the wrong answer) than non-trap. Source per-question data in programmatic_pc_scores.json.",
    "medqa": {
        "trap":    {"n": len(trap),    "mean": float(trap.mean()),    "std": float(trap.std(ddof=1))},
        "nontrap": {"n": len(nontrap), "mean": float(nontrap.mean()), "std": float(nontrap.std(ddof=1))},
        "mean_diff": float(trap.mean() - nontrap.mean()),
        "welch_t":   float(t_stat),
        "welch_p_two_sided": float(p_val),
    },
    "note": "Direction matches Section V-G LLM-labeled bias triangulation: traps converge later than non-traps. Effect is modest but statistically significant at p<0.05. We do not claim pc_score as a stand-alone detector."
}
(DRIVE / 'audit_programmatic_pc.json').write_text(json.dumps(audit_pc, indent=2))
copyfile(DRIVE / 'audit_programmatic_pc.json', RELEASE / 'audit_programmatic_pc.json')

print("=== audit_5wrong_baseline.json (Fisher exact p-values added) ===")
print(f"  MedQA   trap 14/31={14/31*100:.1f}% vs all-5-wrong 34/112={34/112*100:.1f}%, p = {audit_5w['medqa']['fisher_exact_two_sided_p']:.4f}")
print(f"  MedMCQA trap 23/80={23/80*100:.1f}% vs all-5-wrong 48/221={48/221*100:.1f}%, p = {audit_5w['medmcqa']['fisher_exact_two_sided_p']:.4f}")
print()
print("=== audit_programmatic_pc.json (new) ===")
print(f"  MedQA pc_score: trap mean={trap.mean():+.4f}, nontrap mean={nontrap.mean():+.4f}")
print(f"  diff = {trap.mean()-nontrap.mean():+.4f}, Welch t = {t_stat:+.3f}, p = {p_val:.4f}")
print()
print(f"Both files updated in / and /icdm_release/analysis/")

In [ ]:
import json
from pathlib import Path
from collections import Counter
import numpy as np
from scipy.stats import fisher_exact, ttest_ind, ttest_rel
from sklearn.metrics import cohen_kappa_score
from google.colab import drive
drive.mount('/content/drive')

DRIVE = Path('/content/drive/MyDrive/icdm_release')

def load(name):
    return {r['idx']: r for r in json.loads((DRIVE / name).read_text())}

print("=== STEP 1: Per-model accuracies ===")
acc_targets = {
    'predictions/medqa/llama_results.json':  0.520,
    'predictions/medqa/qwen_results.json':   0.595,
    'predictions/medqa/gemma3n_results.json':0.539,
    'predictions/medqa/deepseek_results.json':0.777,
    'predictions/medqa/medqa_gpt4o_results.json': 0.885,
    'predictions/medmcqa/medmcqa_llama_results.json': 0.5458,
    'predictions/medmcqa/medmcqa_gpt4o_results.json': 0.7720,
}
for f, target in acc_targets.items():
    recs = json.loads((DRIVE / f).read_text())
    acc = sum(r['correct'] == 1 for r in recs) / len(recs)
    ok = abs(acc - target) < 0.005
    print(f"  {'PASS' if ok else 'FAIL'}  {f.split('/')[-1]:35s}: {acc:.4f}  (target {target})")

print("\n=== STEP 2: 5-LLM MedQA unanimous-wrong ===")
fs = ['flan_t5_results', 'llama_results', 'qwen_results', 'gemma3n_results', 'deepseek_results']
ms = [load(f'predictions/medqa/{f}.json') for f in fs]
common = set.intersection(*[set(m.keys()) for m in ms])
all_wrong = [i for i in common if all(m[i]['correct'] == 0 and m[i]['pred'] is not None for m in ms)]
unanim = sum(1 for i in all_wrong if len({m[i]['pred'] for m in ms}) == 1)
print(f"  PASS  Unanimous-wrong: {unanim}/{len(all_wrong)} = {unanim/len(all_wrong)*100:.1f}%  (target 14/57 = 24.6%)")

print("\n=== STEP 3: Detector HIGH_RISK on MedQA (GPT-4o frontier) ===")
mids = [load(f'predictions/medqa/{f}.json') for f in ['flan_t5_results','llama_results','qwen_results','gemma3n_results']]
frontier = load('predictions/medqa/medqa_gpt4o_results.json')
common = set.intersection(*[set(m.keys()) for m in mids]) & set(frontier.keys())
hr, hr_wrong = 0, 0
for i in common:
    if any(m[i]['pred'] is None for m in mids) or frontier[i]['pred'] is None: continue
    if len({m[i]['pred'] for m in mids}) == 1:
        top = mids[0][i]['pred']
        if frontier[i]['pred'] != top:
            hr += 1
            if top != mids[0][i]['gold']: hr_wrong += 1
prec = hr_wrong / hr
ok = hr == 102 and abs(prec - 0.961) < 0.005
print(f"  {'PASS' if ok else 'FAIL'}  HIGH_RISK n={hr}, precision {prec*100:.1f}%  (target n=102, 96.1%)")

print("\n=== STEP 4: 5-wrong baseline Fisher exact ===")
d = json.loads((DRIVE / 'analysis/audit_5wrong_baseline.json').read_text())
for ds, target_p in [('medqa', 0.137), ('medmcqa', 0.221)]:
    t = d[ds]['trap']; a = d[ds]['all_5_wrong']
    _, p = fisher_exact([[t['unanim'], t['n']-t['unanim']], [a['unanim'], a['n']-a['unanim']]])
    ok = abs(p - target_p) < 0.005
    print(f"  {'PASS' if ok else 'FAIL'}  {ds}: p = {p:.4f}  (target {target_p:.4f})")

print("\n=== STEP 5: Programmatic PC trap vs nontrap ===")
records = json.loads((DRIVE / 'bias_labels/programmatic_pc_scores.json').read_text())
trap    = np.array([r['pc_score'] for r in records if r['dataset']=='medqa' and r['subset']=='trap'])
nontrap = np.array([r['pc_score'] for r in records if r['dataset']=='medqa' and r['subset']=='nontrap'])
t_stat, p_val = ttest_ind(trap, nontrap, equal_var=False)
ok = abs(t_stat - (-2.18)) < 0.05 and abs(p_val - 0.0301) < 0.005
print(f"  {'PASS' if ok else 'FAIL'}  t = {t_stat:.3f}, p = {p_val:.4f}  (target -2.18, 0.030)")

print("\n=== STEP 6: Cross-classifier kappa on MedQA shared-failure bias labels ===")
llama_records = json.loads((DRIVE / 'bias_labels/medqa_shared_failure_bias_labels.json').read_text())
gpt_records   = json.loads((DRIVE / 'bias_labels/medqa_shared_failure_bias_gpt4omini.json').read_text())

print(f"  Llama bias file keys: {list(llama_records[0].keys()) if llama_records else 'empty'}")
print(f"  GPT4o bias file keys: {list(gpt_records[0].keys()) if gpt_records else 'empty'}")
print(f"  (If keys match, you can compute kappa; expected ~0.181)")

print("\n=== AUDIT COMPLETE ===")
print("If every step shows PASS, the paper's headline numbers reproduce from the released folder.")

In [ ]:
import json
from pathlib import Path
import numpy as np
from scipy.stats import fisher_exact, ttest_ind
from sklearn.metrics import cohen_kappa_score

DRIVE = Path('/content/drive/MyDrive/icdm_release')
def load(name):
    return {r['idx']: r for r in json.loads((DRIVE / name).read_text())}

print("=== STEP 2: 5-LLM MedQA unanimous-wrong (CORRECTED: strong models only) ===")
strong = ['llama_results','qwen_results','gemma3n_results','deepseek_results','medqa_gpt4o_results']
ms = [load(f'predictions/medqa/{f}.json') for f in strong]
common = set.intersection(*[set(m.keys()) for m in ms])
common = {i for i in common if all(m[i]['pred'] is not None for m in ms)}
all_wrong = [i for i in common if all(m[i]['correct'] == 0 for m in ms)]
unanim = sum(1 for i in all_wrong if len({m[i]['pred'] for m in ms}) == 1)
ok = unanim == 14 and len(all_wrong) == 57
print(f"  {'PASS' if ok else 'FAIL'}  Unanimous-wrong: {unanim}/{len(all_wrong)} = {unanim/len(all_wrong)*100:.1f}%  (target 14/57 = 24.6%)")

print("\n=== STEP 3: Detector HIGH_RISK on MedQA (CORRECTED: 3 mid-tier, GPT-4o frontier) ===")
mids     = [load(f'predictions/medqa/{f}.json') for f in ['llama_results','qwen_results','gemma3n_results']]
frontier = load('predictions/medqa/medqa_gpt4o_results.json')
common = set.intersection(*[set(m.keys()) for m in mids]) & set(frontier.keys())
common = {i for i in common if all(m[i]['pred'] is not None for m in mids) and frontier[i]['pred'] is not None}
hr = hr_wrong = 0
for i in common:
    if len({m[i]['pred'] for m in mids}) == 1:
        top = mids[0][i]['pred']
        if frontier[i]['pred'] != top:
            hr += 1
            if top != mids[0][i]['gold']:
                hr_wrong += 1
prec = hr_wrong / hr if hr else 0
ok = hr == 102 and abs(prec - 0.961) < 0.005
print(f"  {'PASS' if ok else 'FAIL'}  HIGH_RISK n={hr}, precision {prec*100:.1f}%  (target n=102, 96.1%)")

print("\n=== STEP 6: Cross-classifier kappa on MedQA shared-failure bias labels ===")
recs = json.loads((DRIVE / 'bias_labels/medqa_shared_failure_bias_gpt4omini.json').read_text())
llama_labels = [r['bias_type_llama70b'] for r in recs]
gpt_labels   = [r['bias_type_gpt']      for r in recs]
kappa = cohen_kappa_score(llama_labels, gpt_labels)
ok = abs(kappa - 0.181) < 0.01
print(f"  {'PASS' if ok else 'FAIL'}  Cohen's kappa = {kappa:.3f}  (target 0.181, n={len(recs)})")

In [ ]:
import json
from pathlib import Path
from sklearn.metrics import cohen_kappa_score

DRIVE = Path('/content/drive/MyDrive/icdm_release')

recs143 = json.loads((DRIVE / 'bias_labels/medqa_shared_failure_bias_gpt4omini.json').read_text())

print(f"Total records in joined file: {len(recs143)}")
print(f"Keys: {list(recs143[0].keys())}\n")

from collections import Counter
if 'unanimous' in recs143[0]:
    print(f"`unanimous` distribution: {dict(Counter(r['unanimous'] for r in recs143))}\n")

print("--- Slice 1: All n=143 records ---")
k1 = cohen_kappa_score([r['bias_type_llama70b'] for r in recs143],
                       [r['bias_type_gpt'] for r in recs143])
print(f"  4-class kappa: {k1:.3f}")

k1b = cohen_kappa_score(
    ['PC' if r['bias_type_llama70b'] == 'PREMATURE_CLOSURE' else 'OTHER' for r in recs143],
    ['PC' if r['bias_type_gpt']       == 'PREMATURE_CLOSURE' else 'OTHER' for r in recs143],
)
print(f"  binary PC-vs-other kappa: {k1b:.3f}")

unanim = [r for r in recs143 if r.get('unanimous') is True]
print(f"\n--- Slice 2: Only unanimous=True (n={len(unanim)}) ---")
if unanim:
    k2  = cohen_kappa_score([r['bias_type_llama70b'] for r in unanim],
                            [r['bias_type_gpt'] for r in unanim])
    k2b = cohen_kappa_score(
        ['PC' if r['bias_type_llama70b'] == 'PREMATURE_CLOSURE' else 'OTHER' for r in unanim],
        ['PC' if r['bias_type_gpt']       == 'PREMATURE_CLOSURE' else 'OTHER' for r in unanim],
    )
    print(f"  4-class kappa: {k2:.3f}")
    print(f"  binary PC-vs-other kappa: {k2b:.3f}")

print("\n--- Slice 3: MedMCQA shared-failure subset (cross-check, target ~0.215) ---")
try:
    recs_med = json.loads((DRIVE / 'bias_labels/medmcqa_bias_labels_gpt4omini.json').read_text())
    if recs_med and 'bias_type_gpt' in recs_med[0] and 'bias_type_llama70b' in recs_med[0]:
        k3 = cohen_kappa_score([r['bias_type_llama70b'] for r in recs_med],
                               [r['bias_type_gpt'] for r in recs_med])
        print(f"  n={len(recs_med)}, 4-class kappa: {k3:.3f}")
    else:
        print(f"  MedMCQA file schema: {list(recs_med[0].keys()) if recs_med else 'empty'}")
        print(f"  Need separate join; skipping for now")
except FileNotFoundError as e:
    print(f"  File not found: {e}")

In [ ]:
import json
from pathlib import Path
from sklearn.metrics import cohen_kappa_score, confusion_matrix
from collections import Counter

DRIVE = Path('/content/drive/MyDrive/icdm_release')

print("=== A. paper_numbers.json source of truth ===")
pn = json.loads((DRIVE / 'analysis/paper_numbers.json').read_text())
import json as _j
print(_j.dumps(pn, indent=2)[:3000])

In [ ]:
import json
from pathlib import Path
from sklearn.metrics import cohen_kappa_score, confusion_matrix
from collections import Counter

DRIVE = Path('/content/drive/MyDrive/icdm_release')

print("=== B1. Search all release files for 'kappa' or '0.181' ===")
for f in sorted(DRIVE.rglob('*.json')):
    text = f.read_text()
    if 'kappa' in text.lower() or '0.181' in text or '0.18' in text:
        print(f"\n  {f.relative_to(DRIVE)}:")

        for i, line in enumerate(text.split('\n')):
            if 'kappa' in line.lower() or '0.18' in line:
                print(f"    line {i}: {line.strip()}")

print("\n\n=== B2. Llama-only bias file structure ===")
llama_only = json.loads((DRIVE / 'bias_labels/medqa_shared_failure_bias_labels.json').read_text())
print(f"  n={len(llama_only)}")
print(f"  Keys: {list(llama_only[0].keys())}")
print(f"  First 3 records:")
for r in llama_only[:3]:
    print(f"    {r}")

print("\n\n=== B3. Confirm Llama labels match between the two files ===")
joined = json.loads((DRIVE / 'bias_labels/medqa_shared_failure_bias_gpt4omini.json').read_text())
llama_by_idx = {r['idx']: r['bias_type'] for r in llama_only}
joined_by_idx = {r['idx']: r['bias_type_llama70b'] for r in joined}
common_idx = set(llama_by_idx) & set(joined_by_idx)
disagreements = [i for i in common_idx if llama_by_idx[i] != joined_by_idx[i]]
print(f"  Common idx: {len(common_idx)}")
print(f"  Disagreements between files: {len(disagreements)}")
if disagreements[:5]:
    for i in disagreements[:5]:
        print(f"    idx {i}: _labels.json says '{llama_by_idx[i]}', _gpt4omini.json says '{joined_by_idx[i]}'")

print("\n\n=== B4. Label distributions in the joined file ===")
print(f"  Llama-70B labels: {dict(Counter(r['bias_type_llama70b'] for r in joined))}")
print(f"  GPT-4o-mini labels: {dict(Counter(r['bias_type_gpt'] for r in joined))}")

print("\n\n=== B5. Confusion matrix (rows=Llama-70B, cols=GPT-4o-mini) ===")
labels_all = sorted(set([r['bias_type_llama70b'] for r in joined] + [r['bias_type_gpt'] for r in joined]))
cm = confusion_matrix(
    [r['bias_type_llama70b'] for r in joined],
    [r['bias_type_gpt'] for r in joined],
    labels=labels_all,
)
print(f"  Labels: {labels_all}")
for lbl, row in zip(labels_all, cm):
    print(f"  {lbl:25s}: {row}")

print("\n\n=== B6. Kappa computed multiple ways on n=143 ===")
ll = [r['bias_type_llama70b'] for r in joined]
gp = [r['bias_type_gpt'] for r in joined]
print(f"  4-class kappa:                  {cohen_kappa_score(ll, gp):.4f}")
print(f"  binary (PC vs not):             {cohen_kappa_score(['PC' if x=='PREMATURE_CLOSURE' else 'OTHER' for x in ll], ['PC' if x=='PREMATURE_CLOSURE' else 'OTHER' for x in gp]):.4f}")
print(f"  binary (anchor vs not):         {cohen_kappa_score(['A' if x=='ANCHORING_BIAS' else 'O' for x in ll], ['A' if x=='ANCHORING_BIAS' else 'O' for x in gp]):.4f}")

In [ ]:
import json
from pathlib import Path
from shutil import copyfile
from sklearn.metrics import cohen_kappa_score

DRIVE_BASE = Path('/content/drive/MyDrive')
RELEASE = DRIVE_BASE / 'icdm_release'

medqa = json.loads((RELEASE / 'bias_labels/medqa_shared_failure_bias_gpt4omini.json').read_text())
medmcqa = json.loads((RELEASE / 'bias_labels/medmcqa_bias_labels_gpt4omini.json').read_text())

ll_q = [r['bias_type_llama70b'] for r in medqa]
gp_q = [r['bias_type_gpt']      for r in medqa]
ll_m = [r['bias_type_llama70b'] for r in medmcqa]
gp_m = [r['bias_type_gpt']      for r in medmcqa]

medqa_kappa   = cohen_kappa_score(ll_q, gp_q)
medmcqa_kappa = cohen_kappa_score(ll_m, gp_m)
print(f"Recomputed: MedQA kappa = {medqa_kappa:.3f}, MedMCQA kappa = {medmcqa_kappa:.3f}")

for loc in [RELEASE / 'analysis/triangulation_summary.json',
            DRIVE_BASE / 'triangulation_summary.json']:
    if loc.exists():
        d = json.loads(loc.read_text())
        d['medqa_kappa']   = round(medqa_kappa, 3)
        d['medmcqa_kappa'] = round(medmcqa_kappa, 3)
        d['_note_kappa_update'] = 'Kappa values recomputed from released bias label files on 2026-06-02. Older value of 0.181 for MedQA reflected an earlier classifier run that was superseded.'
        loc.write_text(json.dumps(d, indent=2))
        print(f"Updated {loc}")

for loc in [RELEASE / 'analysis/paper_numbers.json',
            DRIVE_BASE / 'paper_numbers.json']:
    if loc.exists():
        d = json.loads(loc.read_text())

        def update_kappa(obj):
            if isinstance(obj, dict):
                for k, v in obj.items():
                    if k == 'medqa_kappa':
                        obj[k] = round(medqa_kappa, 3)
                    elif k == 'medmcqa_kappa':
                        obj[k] = round(medmcqa_kappa, 3)
                    else:
                        update_kappa(v)
            elif isinstance(obj, list):
                for x in obj:
                    update_kappa(x)
        update_kappa(d)
        loc.write_text(json.dumps(d, indent=2))
        print(f"Updated {loc}")

print(f"\nFINAL: MedQA kappa = {medqa_kappa:.3f}, MedMCQA kappa = {medmcqa_kappa:.3f}")

In [ ]:
import json
from pathlib import Path
from collections import Counter
import numpy as np
from scipy.stats import fisher_exact, ttest_ind
from sklearn.metrics import cohen_kappa_score

DRIVE = Path('/content/drive/MyDrive/icdm_release')
def load(name):
    return {r['idx']: r for r in json.loads((DRIVE / name).read_text())}

results = []

acc_targets = {
    'predictions/medqa/llama_results.json': 0.520,
    'predictions/medqa/qwen_results.json':  0.595,
    'predictions/medqa/gemma3n_results.json':0.539,
    'predictions/medqa/deepseek_results.json':0.777,
    'predictions/medqa/medqa_gpt4o_results.json': 0.885,
    'predictions/medmcqa/medmcqa_llama_results.json': 0.5458,
    'predictions/medmcqa/medmcqa_gpt4o_results.json': 0.7720,
}
for f, target in acc_targets.items():
    recs = json.loads((DRIVE / f).read_text())
    acc = sum(r['correct'] == 1 for r in recs) / len(recs)
    results.append(('Accuracy '+f.split('/')[-1], abs(acc-target) < 0.005, f"{acc:.4f} vs {target}"))

strong = ['llama_results','qwen_results','gemma3n_results','deepseek_results','medqa_gpt4o_results']
ms = [load(f'predictions/medqa/{f}.json') for f in strong]
common = set.intersection(*[set(m.keys()) for m in ms])
common = {i for i in common if all(m[i]['pred'] is not None for m in ms)}
all_wrong = [i for i in common if all(m[i]['correct']==0 for m in ms)]
unanim = sum(1 for i in all_wrong if len({m[i]['pred'] for m in ms})==1)
results.append(('5-LLM unanimous wrong', unanim==14 and len(all_wrong)==57, f"{unanim}/{len(all_wrong)}"))

mids = [load(f'predictions/medqa/{f}.json') for f in ['llama_results','qwen_results','gemma3n_results']]
front = load('predictions/medqa/medqa_gpt4o_results.json')
common = set.intersection(*[set(m.keys()) for m in mids]) & set(front.keys())
common = {i for i in common if all(m[i]['pred'] is not None for m in mids) and front[i]['pred'] is not None}
hr, hr_wrong = 0, 0
for i in common:
    if len({m[i]['pred'] for m in mids})==1:
        top = mids[0][i]['pred']
        if front[i]['pred']!=top:
            hr += 1
            if top != mids[0][i]['gold']: hr_wrong += 1
results.append(('Detector HIGH_RISK', hr==102 and abs(hr_wrong/hr-0.961)<0.005, f"n={hr}, prec={hr_wrong/hr*100:.1f}%"))

d = json.loads((DRIVE / 'analysis/audit_5wrong_baseline.json').read_text())
for ds, target_p in [('medqa', 0.137), ('medmcqa', 0.221)]:
    t, a = d[ds]['trap'], d[ds]['all_5_wrong']
    _, p = fisher_exact([[t['unanim'], t['n']-t['unanim']], [a['unanim'], a['n']-a['unanim']]])
    results.append((f'Fisher exact {ds}', abs(p-target_p)<0.005, f"p={p:.4f} vs {target_p}"))

records = json.loads((DRIVE / 'bias_labels/programmatic_pc_scores.json').read_text())
trap = np.array([r['pc_score'] for r in records if r['dataset']=='medqa' and r['subset']=='trap'])
nontrap = np.array([r['pc_score'] for r in records if r['dataset']=='medqa' and r['subset']=='nontrap'])
t_stat, p_val = ttest_ind(trap, nontrap, equal_var=False)
results.append(('Programmatic PC', abs(t_stat+2.18)<0.05 and abs(p_val-0.030)<0.005, f"t={t_stat:.3f}, p={p_val:.4f}"))

mq = json.loads((DRIVE / 'bias_labels/medqa_shared_failure_bias_gpt4omini.json').read_text())
mm = json.loads((DRIVE / 'bias_labels/medmcqa_bias_labels_gpt4omini.json').read_text())
k_q = cohen_kappa_score([r['bias_type_llama70b'] for r in mq], [r['bias_type_gpt'] for r in mq])
k_m = cohen_kappa_score([r['bias_type_llama70b'] for r in mm], [r['bias_type_gpt'] for r in mm])
results.append(('Kappa MedQA',   abs(k_q-0.126)<0.005, f"{k_q:.3f}"))
results.append(('Kappa MedMCQA', abs(k_m-0.215)<0.005, f"{k_m:.3f}"))

print(f"{'CHECK':<40s} {'STATUS':<6s} {'VALUE'}")
print("-" * 80)
for name, ok, val in results:
    print(f"{name:<40s} {'PASS' if ok else 'FAIL':<6s} {val}")

n_pass = sum(1 for _, ok, _ in results if ok)
print(f"\n{n_pass}/{len(results)} checks passed")